# Actividad 03 — Features Temporales: Codificación Cíclica y Lags

**Fase:** 2 — Ingeniería de Características Multimodal  
**Dominio:** Predicción de producción de limón (Sutil y Dulce), 2016-2025  
**Referencia metodológica:** Actividades 02 y 03 de v1 (`notebooks/fase2/actividad_02_cyclic_time_encoding.ipynb`,
`actividad_03_rezagos_temporales.ipynb`)

---

## Objetivo

Generar sobre ambos datasets maestro v2 (Sutil y Dulce):

1. **Codificación cíclica del mes** (elimina el salto artificial dic-ene).
2. **Rezagos temporales** t-1, t-3, t-6 para la variable objetivo y las variables
   climáticas principales — mismo esquema v1 (timesteps=6).
3. **Decisión documentada** sobre los primeros 6 meses sin historial completo.
4. **Verificación anti-fuga de datos**: ningún lag usa información del futuro.

## Entrada

- `v2_reentrenamiento/data/processed/master_dataset_sutil_v2.csv` (120×18)
- `v2_reentrenamiento/data/processed/master_dataset_dulce_v2.csv` (120×18)

## Salida

- `v2_reentrenamiento/data/processed/master_dataset_sutil_v2_features.csv`
- `v2_reentrenamiento/data/processed/master_dataset_dulce_v2_features.csv`

---


## 1. Configuración inicial


In [ ]:
import os, warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
pd.set_option('display.width', 240)
pd.set_option('display.max_columns', None)

while not os.path.exists('v2_reentrenamiento/data/processed'):
    os.chdir('..')
print('Raiz del proyecto:', os.getcwd())

PROC = 'v2_reentrenamiento/data/processed'
IN_SUTIL  = f'{PROC}/master_dataset_sutil_v2.csv'
IN_DULCE  = f'{PROC}/master_dataset_dulce_v2.csv'
OUT_SUTIL = f'{PROC}/master_dataset_sutil_v2_features.csv'
OUT_DULCE = f'{PROC}/master_dataset_dulce_v2_features.csv'

LAG_STEPS = [1, 3, 6]
# Variable objetivo + 3 variables climáticas principales (por requerimiento).
CLIMA_LAG = ['T2M', 'WS2M', 'PRECTOTCORR']


## 2. Carga y ordenamiento cronológico


In [ ]:
master_sutil = pd.read_csv(IN_SUTIL, encoding='utf-8-sig')
master_dulce = pd.read_csv(IN_DULCE, encoding='utf-8-sig')
print('Sutil:', master_sutil.shape, '| Dulce:', master_dulce.shape)

# Orden cronológico estricto (indispensable para shift() posterior)
for df in (master_sutil, master_dulce):
    df.sort_values(['año', 'mes'], inplace=True)
    df.reset_index(drop=True, inplace=True)

print('Ordenado OK. Primeras 2 filas Sutil:')
print(master_sutil[['año', 'mes', 'produccion_t_sutil']].head(2).to_string(index=False))


## 3. PASO 1 — Codificación cíclica del mes

`mes_sin = sin(2π · mes / 12)` · `mes_cos = cos(2π · mes / 12)`

Diciembre (mes=12) y enero (mes=1) quedan adyacentes en el círculo trigonométrico,
eliminando el salto artificial de la variable `mes` entera.


In [ ]:
def codificacion_ciclica(df):
    df = df.copy()
    df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
    df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)
    return df

master_sutil = codificacion_ciclica(master_sutil)
master_dulce = codificacion_ciclica(master_dulce)
print('Sutil +2 cols:', master_sutil.shape)
print('Dulce +2 cols:', master_dulce.shape)
print()
print('Valores cíclicos (mes 1, 6, 12):')
v = master_sutil[master_sutil['mes'].isin([1, 6, 12])][['mes', 'mes_sin', 'mes_cos']].drop_duplicates('mes')
print(v.sort_values('mes').round(4).to_string(index=False))
# check: seno/coseno de 1 y 12 deben ser adyacentes (12→2π≈0, cos≈1)
print()
print('Adyacencia dic(12)→ene(1):', 'OK' if abs(master_sutil.loc[master_sutil.mes==12,'mes_cos'].iloc[0]-1)<0.01 else 'FALLA')


## 4. PASO 2 — Generación de lags temporales (t-1, t-3, t-6)

Mismo esquema v1 (`df[col].shift(lag)` sobre serie ordenada). La serie es **nacional**
(no hay agrupación por provincia), de modo que el `shift` opera sobre el orden completo
año-mes.

> ⚠️ **Anti-fuga**: los lags se calculan como transformaciones **deterministas del pasado**
> (col.observada en t-k). No se ajusta ningún estimador ni se usan valores futuros.
> La verificación numérica está en la sección 6.


In [ ]:
def generar_lags(df, target):
    lag_cols = [target] + CLIMA_LAG
    etiquetas = []
    for col in lag_cols:
        for lag in LAG_STEPS:
            nc = f'{col}_lag{lag}'
            df[nc] = df[col].shift(lag)
            etiquetas.append(nc)
    return df, etiquetas

master_sutil, lags_sutil = generar_lags(master_sutil, 'produccion_t_sutil')
master_dulce, lags_dulce = generar_lags(master_dulce, 'produccion_t_dulce')
print(f'Sutil shape con lags: {master_sutil.shape}')
print(f'Dulce shape con lags: {master_dulce.shape}')
print(f'Columnas lag generadas ({len(lags_sutil)}):')
print(lags_sutil)


## 5. PASO 3 — Manejo de los primeros 6 meses sin historial


In [ ]:
for tgt, df in [('produccion_t_sutil', master_sutil), ('produccion_t_dulce', master_dulce)]:
    lags = [tgt + '_lag' + str(k) for k in LAG_STEPS] + [c + '_lag6' for c in CLIMA_LAG]
    nan_rows = df[df[lags].isna().any(axis=1)]
    print(f'FILAS CON LAG INCOMPLETO ({tgt}): {len(nan_rows)}')
    print(nan_rows[['año', 'mes'] + lags].to_string(index=False))
    print()


### Decisión documentada — PASO 3

**Estrategia elegida: eliminar las filas (dropna), igual que v1.**

| Opción | Efecto | Veredicto |
|---|---|---|
| El <b>imputar/rellenar</b> (p.ej. con 0 o la media) | Inyecta datos fabricados en el inicio de la serie, corrompiendo la ventana de historia | ❌ Descartado |
| <b>Dejar NaN</b> en las columnas lag | El modelo (LSTM/MLP) no consume NaN; habría que imputarlos más adelante igual | ❌ Descartado |
| <b>Eliminar las filas iniciales</b> sin t-6 | Consistente con v1 (`dropna(subset=lags)`); los lags solo se usan como entrada en los canales de contexto | ✅ **Elegido** |

**Consecuencia sobre el split train/val/test (train=2016-2023):**
- Se eliminan **6 filas** (2016-01 a 2016-06) por no contar con historial de 6 meses previos.
- El entrenamiento efectivo arranca en **2016-07** en lugar de 2016-01 → el train pasa
  a 90 meses en vez de 96 (4 filas de 2016-06 a... exactamente 2016-01..06 = 6 filas).
- La pérdida es **< 5.0%** de la serie y no afecta val (2024) ni test (2025).

> ⚠️ Nota metodológica para etapas posteriores: si el modelo llega a necesitar predicción
> para 2016-temprano (fuera de scope, ya que train/test usan 2016-07+), no habrá lag completo.
> El objetivo futuro se predice desde julio 2016 en adelante, sin pérdida práctica.


In [ ]:
def aplicar_dropna(df, lag_cols):
    antes = len(df)
    dfc = df.dropna(subset=lag_cols).reset_index(drop=True)
    print(f'Filas antes del dropna: {antes} | después: {len(dfc)} | eliminadas: {antes - len(dfc)}')
    assert dfc[lag_cols].isna().sum().sum() == 0, 'Quedan NaNs en lags!'
    return dfc

master_sutil = aplicar_dropna(master_sutil, lags_sutil)
master_dulce = aplicar_dropna(master_dulce, lags_dulce)
print('Sutil:', master_sutil.shape, '| Dulce:', master_dulce.shape)
print('Rango final Sutil:', master_sutil['año'].min(), master_sutil['año'].max())
print('Primera fila Sutil:')
print(master_sutil[['año', 'mes', 'produccion_t_sutil']].head(1).to_string(index=False))


## 6. Verificación anti-fuga de datos (ningún lag usa el futuro)

Comprobación por inspección y por cálculo en la fila de **enero 2020**:
los lags `_lag1`, `_lag3`, `_lag6` deben ser exactamente los valores de
**2019-12, 2019-10 y 2019-07** respectivamente.


In [ ]:
fila_2020 = master_sutil[master_sutil['año'] == 2020].head(1)
i = fila_2020.index[0]
print('FILA COMPROBADA: enero 2020 (índice', i, ')')
print('  produccion_t_sutil actual        :', fila_2020['produccion_t_sutil'].iloc[0])
print('  produccion_t_sutil_lag1 (2019-12):', fila_2020['produccion_t_sutil_lag1'].iloc[0])
print('  produccion_t_sutil_lag3 (2019-10):', fila_2020['produccion_t_sutil_lag3'].iloc[0])
print('  produccion_t_sutil_lag6 (2019-07):', fila_2020['produccion_t_sutil_lag6'].iloc[0])
print()
esperado = {
    'produccion_t_sutil_lag1': master_sutil.loc[i - 1, 'produccion_t_sutil'],
    'produccion_t_sutil_lag3': master_sutil.loc[i - 3, 'produccion_t_sutil'],
    'produccion_t_sutil_lag6': master_sutil.loc[i - 6, 'produccion_t_sutil'],
    'T2M_lag6':               master_sutil.loc[i - 6, 'T2M'],
}
for c, v in esperado.items():
    ok = abs(master_sutil.loc[i, c] - v) < 1e-9
    print(f'  {c:26s} == valor en fila t-{c.split("_lag")[1]:s}: ', 'OK' if ok else 'FALLA')
print()
print('=> Ninguna celda lag apunta al pasado estricto de su variable (t-k).')
print('=> Conclusión: sin información del futuro respecto a la fila actual.')


### Verificación programática completa


In [ ]:
def verificar_lags(df, lag_cols, target):
    fracasos = []
    for j in range(6, len(df)):
        for c in lag_cols:
            lag = int(c.rsplit('_lag', 1)[1])
            base = c.replace(f'_lag{lag}', '')
            valor = df.loc[j, c]
            esperado = df.loc[j - lag, base] if j - lag >= 0 else np.nan
            if pd.notna(valor) and pd.notna(esperado) and abs(valor - esperado) > 1e-9:
                fracasos.append((j, c))
    print(f'{target}: {len(df)} filas verificadas | problemas: {len(fracasos)}')
    if fracasos: print('FALLOS:', fracasos[:10])
    return len(fracasos) == 0

ok_s = verificar_lags(master_sutil, lags_sutil, 'sutil')
ok_d = verificar_lags(master_dulce, lags_dulce, 'dulce')
print()
print('VERIFICACIÓN ANTI-FUGA:', 'PASA ✓' if (ok_s and ok_d) else 'FALLA ✗')
print('-> Cada celda _lagk contiene EXACTAMENTE el valor de la misma variable en t-k (pasado estricto).')


## 7. PASO 4 — Guardar y validar


In [ ]:
os.makedirs(PROC, exist_ok=True)
master_sutil.to_csv(OUT_SUTIL, index=False, encoding='utf-8-sig')
master_dulce.to_csv(OUT_DULCE, index=False, encoding='utf-8-sig')
print('Guardado:', OUT_SUTIL)
print('Guardado:', OUT_DULCE)


In [ ]:
def validar(df, nombre):
    nulos = df.isna().sum()
    print('=' * 70)
    print(nombre, '|', df.shape)
    print('-' * 70)
    print('1) Filas:', len(df), '(esperado 114 = 120 - 6 iniciales)')
    print('   Rango:', f'{df["año"].min()}-{df["mes"].min()} -> {df["año"].max()}-{df["mes"].max()}')
    print('2) Nulos por columna:')
    print(nulos.to_string() if nulos.sum() else '   (ninguno en ninguna columna)')
    print('3) Número de columnas:', len(df.columns))
    return df

validar(master_sutil, 'MAESTRO SUTIL v2 + features')
validar(master_dulce, 'MAESTRO DULCE v2 + features')


## Primeras 10 filas — inspección visual del manejo de lags iniciales


In [ ]:
print('=== SUTIL — primeras 10 filas (año, mes, mes_sin/cos, prod y sus lags) ===')
cols_mostrar = ['año', 'mes', 'mes_sin', 'mes_cos', 'produccion_t_sutil',
                'produccion_t_sutil_lag1', 'produccion_t_sutil_lag3',
                'produccion_t_sutil_lag6', 'T2M', 'WS2M', 'PRECTOTCORR',
                'T2M_lag6', 'avg_sentiment']
print(master_sutil[cols_mostrar].head(10).round(4).to_string(index=False))
print()
print('=> La fila 2016-07 (primera) toma sus lags de 2016-06, 2016-04 y 2016-01 (existentes).')
print('=> Las filas 2016-01..2016-06 fueron eliminadas por dropna (PASO 3).')


In [ ]:
print('=== DULCE — primeras 10 filas ===')
cols_mostrar = ['año', 'mes', 'produccion_t_dulce',
                'produccion_t_dulce_lag1', 'produccion_t_dulce_lag3',
                'produccion_t_dulce_lag6', 'T2M', 'PRECTOTCORR']
print(master_dulce[cols_mostrar].head(10).round(4).to_string(index=False))
print()
print('=== FIN ACTIVIDAD 03 (v2) ===')
